# EN3150 Assignment 03 — Shanuka's Notebook
## SOTA Transfer Learning: MobileNetV2 & ShuffleNetV2 on RealWaste 64×64

**Team:** Outliers | **Author:** Shanuka (`@ashanuk`)

This notebook covers:
- **C05** — SOTA model wrappers (MobileNetV2, ShuffleNetV2)
- **C12** — Fine-tuning both models for 20 epochs on RealWaste
- **C14** — Test evaluation, confusion matrices, precision & recall

> ⚠️ **Runtime → Change runtime type → T4 GPU** before running!

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# ── Clone the team repo ───────────────────────────────────────────────────
import os

REPO_URL    = 'https://github.com/yumyum-web/pattern-ass-03.git'
BRANCH      = 'feature/sota-transfer'
REPO_DIR    = '/content/pattern-ass-03'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
# torch/torchvision are already installed on Colab; install the rest
!pip install -q scikit-learn tqdm ucimlrepo torchinfo

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} — device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2. Dataset — RealWaste 64×64 (UCI ID: 908)

In [ ]:
# ── Load data pipeline ────────────────────────────────────────────────────
import sys
sys.path.insert(0, REPO_DIR)

from src.data import get_realwaste_dataloaders, CLASSES, IDX_TO_CLASS

train_loader, val_loader, test_loader, class_to_idx = get_realwaste_dataloaders(
    data_dir='data',
    batch_size=32,
    image_size=64,
    seed=42,
    num_workers=2,
    download=True,
)

total = len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset)
print(f'Classes  : {CLASSES}')
print(f'Train    : {len(train_loader.dataset):,} ({len(train_loader.dataset)/total*100:.1f}%)')
print(f'Val      : {len(val_loader.dataset):,} ({len(val_loader.dataset)/total*100:.1f}%)')
print(f'Test     : {len(test_loader.dataset):,} ({len(test_loader.dataset)/total*100:.1f}%)')

# Verify batch shape
imgs, labels = next(iter(train_loader))
print(f'Batch    : images {tuple(imgs.shape)}, labels {tuple(labels.shape)}')
assert imgs.shape[1:] == (3, 64, 64), 'Unexpected image shape!'
print('✅ Data pipeline verified')

## 3. SOTA Model Wrappers — C05
> **Commit:** `feat(sota): create MobileNetV2 and ShuffleNetV2 transfer learning wrappers`

In [ ]:
# ── Load SOTA wrappers ────────────────────────────────────────────────────
from src.models.sota_models import (
    get_mobilenet_v2,
    get_shufflenet_v2,
    count_parameters,
    get_model_size_mb,
    print_model_summary,
)

mobilenet  = get_mobilenet_v2(num_classes=9, pretrained=True)
shufflenet = get_shufflenet_v2(num_classes=9, pretrained=True, width_mult='1_0')

print_model_summary(mobilenet,  'MobileNetV2')
print_model_summary(shufflenet, 'ShuffleNetV2 ×1.0')

# Quick forward-pass sanity check
dummy = torch.zeros(2, 3, 64, 64)
with torch.no_grad():
    out_mn = mobilenet(dummy)
    out_sn = shufflenet(dummy)
assert out_mn.shape == (2, 9) and out_sn.shape == (2, 9)
print('\n✅ C05: Both SOTA wrappers verified (output shape 2×9)')

## 4. Fine-Tuning — C12
> **Commit:** `test(sota): fine-tune MobileNetV2 and ShuffleNetV2 on RealWaste 64x64`

- **Optimiser:** Adam (lr=0.001, weight_decay=1e-4)
- **Scheduler:** CosineAnnealingLR (T_max=20)
- **Loss:** CrossEntropyLoss
- **Epochs:** 20 per model

In [ ]:
# ── Training helpers ──────────────────────────────────────────────────────
import time
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS       = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHTS_DIR  = 'weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs('figures', exist_ok=True)

print(f'Training on: {DEVICE}')


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, lbls)
            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == lbls).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


def train_model(model, name, save_path):
    model = model.to(DEVICE)
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc   = 0.0

    print(f'\n{"─"*62}')
    print(f'  Training: {name}')
    print(f'{"─"*62}')
    print(f'  {"Ep":>3}  {"TrainLoss":>10}  {"TrainAcc":>9}  {"ValLoss":>9}  {"ValAcc":>8}  {"Time":>6}')

    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = eval_epoch(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)

        print(f'  {ep:>3}  {tl:>10.4f}  {ta*100:>8.2f}%  {vl:>9.4f}  {va*100:>7.2f}%  {time.time()-t0:>5.1f}s')

        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), save_path)

    print(f'\n  ✅ Best val acc: {best_acc*100:.2f}% — saved to {save_path}')
    return model, history

In [ ]:
# ── Train MobileNetV2 (≈ 10–15 min on T4 GPU) ─────────────────────────────
mobilenet, hist_mn = train_model(
    get_mobilenet_v2(num_classes=9, pretrained=True),
    name      = 'MobileNetV2',
    save_path = f'{WEIGHTS_DIR}/mobilenet_v2.pth',
)

In [ ]:
# ── Train ShuffleNetV2 ×1.0 (≈ 10–15 min on T4 GPU) ──────────────────────
shufflenet, hist_sn = train_model(
    get_shufflenet_v2(num_classes=9, pretrained=True, width_mult='1_0'),
    name      = 'ShuffleNetV2-1.0×',
    save_path = f'{WEIGHTS_DIR}/shufflenet_v2.pth',
)

In [ ]:
# ── Plot training curves and save figure ──────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

histories   = {'MobileNetV2': hist_mn, 'ShuffleNetV2-1.0×': hist_sn}
model_names = list(histories.keys())
epochs      = range(1, EPOCHS + 1)

fig, axes = plt.subplots(len(model_names), 2, figsize=(14, 5 * len(model_names)))

for row, name in enumerate(model_names):
    h = histories[name]
    ax_l, ax_a = axes[row]

    ax_l.plot(epochs, h['train_loss'], label='Train', color='#2196F3', lw=1.8)
    ax_l.plot(epochs, h['val_loss'],   label='Val',   color='#F44336', lw=1.8, ls='--')
    ax_l.set_title(f'{name} — Loss',     fontweight='bold')
    ax_l.set_xlabel('Epoch'); ax_l.set_ylabel('Cross-Entropy Loss')
    ax_l.legend(); ax_l.grid(alpha=0.3)

    ax_a.plot(epochs, [a*100 for a in h['train_acc']], label='Train', color='#4CAF50', lw=1.8)
    ax_a.plot(epochs, [a*100 for a in h['val_acc']],   label='Val',   color='#FF9800', lw=1.8, ls='--')
    ax_a.set_title(f'{name} — Accuracy', fontweight='bold')
    ax_a.set_xlabel('Epoch'); ax_a.set_ylabel('Accuracy (%)')
    ax_a.set_ylim(0, 100); ax_a.legend(); ax_a.grid(alpha=0.3)

fig.suptitle('SOTA Transfer Learning — Training Curves\nRealWaste 64×64', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('figures/sota_loss_curves.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('✅ C12: Training complete — figures/sota_loss_curves.pdf saved')

## 5. Evaluation on Test Set — C14
> **Commit:** `feat(eval): evaluate SOTA models on test set with confusion matrices and precision-recall`

In [ ]:
# ── Load metrics module ───────────────────────────────────────────────────
from src.utils.metrics import (
    collect_predictions,
    compute_metrics,
    print_metrics,
    save_confusion_matrices,
    CLASSES as METRIC_CLASSES,
)

model_configs = [
    {'name': 'MobileNetV2',       'model': mobilenet,  'path': f'{WEIGHTS_DIR}/mobilenet_v2.pth'},
    {'name': 'ShuffleNetV2-1.0×', 'model': shufflenet, 'path': f'{WEIGHTS_DIR}/shufflenet_v2.pth'},
]

confusion_data = []
summary_rows   = []

for cfg in model_configs:
    # Load best checkpoint
    cfg['model'].load_state_dict(torch.load(cfg['path'], map_location=DEVICE))
    cfg['model'] = cfg['model'].to(DEVICE)

    preds, targets = collect_predictions(cfg['model'], test_loader, DEVICE)
    print_metrics(preds, targets, model_name=cfg['name'])

    acc, prec, rec = compute_metrics(preds, targets)
    trainable, _   = count_parameters(cfg['model'])
    size_mb        = get_model_size_mb(cfg['model'])

    confusion_data.append((cfg['name'], preds, targets))
    summary_rows.append({'Model': cfg['name'], 'Params': trainable,
                         'Size(MB)': size_mb, 'TestAcc': f"{acc*100:.2f}%",
                         'MacroPrec': f"{prec*100:.2f}%", 'MacroRec': f"{rec*100:.2f}%"})

In [ ]:
# ── Save confusion matrices ───────────────────────────────────────────────
save_confusion_matrices(
    confusion_data,
    save_path='figures/confusion_matrices.pdf',
)

# Also display inline
from IPython.display import display
import matplotlib.pyplot as plt
from src.utils.metrics import plot_confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for idx, (name, preds, targets) in enumerate(confusion_data):
    plot_confusion_matrix(preds, targets, model_name=name, ax=axes[idx])
plt.suptitle('Normalised Confusion Matrices — SOTA Transfer Learning\nRealWaste 64×64 Test Set',
             fontweight='bold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print('✅ C14: Confusion matrices saved to figures/confusion_matrices.pdf')

In [ ]:
# ── Final comparison table ────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(summary_rows)
print('\n=== SOTA MODEL COMPARISON TABLE ===')
print(df.to_markdown(index=False))
display(df)

## 6. Download Outputs
Download the generated figures and model weights to your local machine, then commit them.

In [ ]:
# ── Download all outputs ──────────────────────────────────────────────────
from google.colab import files
import glob

output_files = (
    glob.glob('figures/*.pdf') +
    glob.glob('weights/*.pth')
)

print(f'Downloading {len(output_files)} files:')
for f in output_files:
    print(f'  {f}')
    files.download(f)

## 7. Git Commit Instructions

After downloading the outputs, run these commands locally in `d:\Projects\pattern-ass-03`:

```bash
# C12 — Fine-tuning results
git add experiments/train_sota.py src/utils/metrics.py figures/sota_loss_curves.pdf weights/mobilenet_v2.pth weights/shufflenet_v2.pth
git commit -m "test(sota): fine-tune MobileNetV2 and ShuffleNetV2 on RealWaste 64x64"

# C14 — Evaluation & confusion matrices
git add figures/confusion_matrices.pdf
git commit -m "feat(eval): evaluate SOTA models on test set with confusion matrices and precision-recall"

# Push your branch
git push origin feature/sota-transfer
```